# UFC fighter statistics database

This notebook uses the public UFC API repository by `eminbustun/UFC_API` to build a local SQLite database of UFC fighters whose fighter-statistics records appear complete.

The API base URL used here is:

```text
https://ufc-api-theta.vercel.app/mma-api/fighters
```

The notebook will:

1. fetch every page of fighter records;
2. normalize the JSON into a `pandas` DataFrame;
3. define and apply a “complete statistics” filter;
4. save the filtered fighter table to SQLite;
5. run a few sanity-check queries.

> Note: the API’s source code says that routes using fighter ids should use MongoDB `_id`, not `fighter_id`. This notebook keeps both when available.


In [3]:
# If needed, uncomment this in a fresh environment:
# pip install requests pandas tqdm

import json
import math
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import pandas as pd
import requests
from tqdm.auto import tqdm


In [4]:
API_BASE_URL = "https://ufc-api-theta.vercel.app/mma-api/fighters"

# Local outputs
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RAW_JSON_PATH = DATA_DIR / "ufc_fighters_raw.json"
CSV_PATH = DATA_DIR / "ufc_fighters_complete_stats.csv"
SQLITE_PATH = DATA_DIR / "ufc_fighters.sqlite"

# Be kind to the API.
PAGE_LIMIT = 100
REQUEST_TIMEOUT = 30
REQUEST_SLEEP_SECONDS = 0.15


## Helper functions

The list endpoint returns a JSON object with a `fighters` list and pagination metadata including `hasNextPage`, `nextPage`, and `lastPage`. The helper below follows those pages until the API says there is no next page.


In [5]:
def get_json(url: str, params: Optional[Dict[str, Any]] = None, *, retries: int = 3) -> Dict[str, Any]:
    """GET JSON from the API with simple retry behavior."""
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(1.5 * attempt)
            else:
                raise RuntimeError(f"Failed after {retries} attempts: {url} params={params}") from last_error


def fetch_all_fighters(limit: int = PAGE_LIMIT) -> List[Dict[str, Any]]:
    """Fetch all fighter records from /mma-api/fighters using the paginated endpoint."""
    page = 1
    fighters: List[Dict[str, Any]] = []
    last_page = None

    # First request determines pagination.
    payload = get_json(API_BASE_URL, params={"limit": limit, "page": page})
    if not payload.get("success", False):
        raise RuntimeError(f"API returned unsuccessful payload: {payload}")

    page_fighters = payload.get("fighters", [])
    fighters.extend(page_fighters)
    last_page = payload.get("lastPage")
    has_next = payload.get("hasNextPage", False)
    next_page = payload.get("nextPage")

    if last_page is None:
        pbar = tqdm(disable=True)
    else:
        pbar = tqdm(total=max(last_page - 1, 0), desc="Fetching remaining pages")

    while has_next and next_page:
        time.sleep(REQUEST_SLEEP_SECONDS)
        payload = get_json(API_BASE_URL, params={"limit": limit, "page": next_page})
        if not payload.get("success", False):
            raise RuntimeError(f"API returned unsuccessful payload on page {next_page}: {payload}")
        fighters.extend(payload.get("fighters", []))
        has_next = payload.get("hasNextPage", False)
        next_page = payload.get("nextPage")
        pbar.update(1)

    pbar.close()
    return fighters


## Fetch and inspect the raw fighter records

This cell writes a timestamped raw JSON snapshot to `data/ufc_fighters_raw.json` so that later analysis can be reproduced even if the API changes.


In [6]:
fighters_raw = fetch_all_fighters(limit=PAGE_LIMIT)

snapshot = {
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "api_base_url": API_BASE_URL,
    "n_records": len(fighters_raw),
    "fighters": fighters_raw,
}
RAW_JSON_PATH.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")

print(f"Fetched {len(fighters_raw):,} fighter records.")
print(f"Saved raw snapshot to {RAW_JSON_PATH}")


Fetching remaining pages: 100%|██████████| 42/42 [00:11<00:00,  3.80it/s]

Fetched 4,262 fighter records.
Saved raw snapshot to data/ufc_fighters_raw.json


In [7]:
# Peek at the first record to verify the shape of the data.
fighters_raw[0] if fighters_raw else {}


{'_id': '67533c6e7a2ac5f7e09aec69',
 'fighter_id': '2d077e22886298be',
 'name': 'Wuziazibieke Jiahefu',
 'height': '5\' 9"',
 'weight': '145 lbs.',
 'reach': '70"',
 'stance': 'Orthodox',
 'dob': 'Apr 09, 1990',
 'n_win': 29,
 'n_loss': 12,
 'n_draw': 1,
 'sig_str_land_pM': 0.41,
 'sig_str_land_pct': 0.5,
 'sig_str_abs_pM': 2.84,
 'sig_str_def_pct': 0.41,
 'td_avg': 6.08,
 'td_land_pct': 0.5,
 'td_def_pct': 0,
 'sub_avg': 0,
 '__v': 0}

## Normalize and define “complete statistics”

The fighter model in the API source includes the following statistics fields:

- `n_win`, `n_loss`, `n_draw`
- `sig_str_land_pM`, `sig_str_land_pct`, `sig_str_abs_pM`, `sig_str_def_pct`
- `td_avg`, `td_land_pct`, `td_def_pct`
- `sub_avg`

For “complete statistics,” this notebook requires all of those statistic fields to be present and non-missing. I also include basic profile fields—`name`, `height`, `weight`, `reach`, `stance`, and `dob`—in the completeness check. If you want “complete statistics” to mean only the numeric fight/stat fields, remove the profile fields from `REQUIRED_COMPLETE_FIELDS` below.


In [13]:
df = pd.json_normalize(fighters_raw)

# Ensure expected columns exist even if the API omits one in the future.
EXPECTED_FIELDS = [
    "_id", "fighter_id", "name", "height", "weight", "reach", "stance", "dob",
    "n_win", "n_loss", "n_draw",
    "sig_str_land_pM", "sig_str_land_pct", "sig_str_abs_pM", "sig_str_def_pct",
    "td_avg", "td_land_pct", "td_def_pct", "sub_avg",
]
for col in EXPECTED_FIELDS:
    if col not in df.columns:
        df[col] = pd.NA

# Treat empty strings and common placeholder strings as missing.
MISSING_STRINGS = {"", "--", "-", "N/A", "NA", "NaN", "nan", "None", "null"}
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].map(lambda x: pd.NA if isinstance(x, str) and x.strip() in MISSING_STRINGS else x)

NUMERIC_FIELDS = [
    "n_win", "n_loss", "n_draw",
    "sig_str_land_pM", "sig_str_land_pct", "sig_str_abs_pM", "sig_str_def_pct",
    "td_avg", "td_land_pct", "td_def_pct", "sub_avg",
]
for col in NUMERIC_FIELDS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

PROFILE_FIELDS = ["name", "height", "weight", "reach", "stance", "dob"]
# REQUIRED_COMPLETE_FIELDS = PROFILE_FIELDS + NUMERIC_FIELDS
REQUIRED_COMPLETE_FIELDS = ["name", "height", "weight"] + NUMERIC_FIELDS

complete_mask = df[REQUIRED_COMPLETE_FIELDS].notna().all(axis=1)
df_complete = df.loc[complete_mask, EXPECTED_FIELDS].copy()

print(f"All fighter records: {len(df):,}")
print(f"Complete-stat fighter records: {len(df_complete):,}")
print(f"Dropped for missing at least one required field: {len(df) - len(df_complete):,}")

df_complete.head()


All fighter records: 4,262
Complete-stat fighter records: 3,955
Dropped for missing at least one required field: 307


,_id,fighter_id,name,height,weight,reach,stance,dob,n_win,n_loss,n_draw,sig_str_land_pM,sig_str_land_pct,sig_str_abs_pM,sig_str_def_pct,td_avg,td_land_pct,td_def_pct,sub_avg
0,67533c6e7a2ac5f7e09aec69,2d077e22886298be,Wuziazibieke Jiahefu,"5' 9""",145 lbs.,"70""",Orthodox,"Apr 09, 1990",29,12,1,0.41,0.50,2.84,0.41,6.08,0.50,0.00,0.0
1,67533c6e7a2ac5f7e09aec6a,7debc13b36343605,Cristian Quinonez,"5' 8""",135 lbs.,"70""",Orthodox,"Apr 26, 1996",18,5,0,4.19,0.40,4.55,0.55,1.37,0.37,0.84,0.0
2,67533c6e7a2ac5f7e09aec6b,d29b5c4f22c6357d,Gilbert Yvel,"6' 2""",225 lbs.,"77""",Orthodox,"Jun 30, 1976",39,16,1,1.05,0.47,1.78,0.50,0.00,0.00,0.25,1.0
3,67533c6e7a2ac5f7e09aec70,8667caa0451d245b,Kennedy Nzechukwu,"6' 5""",205 lbs.,"83""",Southpaw,"Jun 13, 1992",13,5,0,5.34,0.47,4.83,0.51,0.58,0.45,0.80,0.2
4,67533c6e7a2ac5f7e09aec6e,7139cd2ae4bf6a29,Takahiro Oba,"5' 8""",200 lbs.,<NA>,Southpaw,<NA>,5,7,1,0.23,0.25,1.82,0.69,0.00,0.00,0.00,0.0


In [14]:
# Optional diagnostic: which fields caused the most records to be dropped?
missing_summary = (
    df[REQUIRED_COMPLETE_FIELDS]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(1)
missing_summary


,missing_count,missing_pct
height,301,7.1
weight,86,2.0
name,0,0.0
n_win,0,0.0
n_loss,0,0.0
n_draw,0,0.0
sig_str_land_pM,0,0.0
sig_str_land_pct,0,0.0
sig_str_abs_pM,0,0.0
sig_str_def_pct,0,0.0


## Save the complete fighter statistics to CSV and SQLite

The SQLite database contains:

- `fighters_complete`: one row per fighter with complete statistics;
- `metadata`: a tiny provenance table recording the API URL, retrieval time, and counts.


In [15]:
df_complete.to_csv(CSV_PATH, index=False)

with sqlite3.connect(SQLITE_PATH) as conn:
    df_complete.to_sql("fighters_complete", conn, if_exists="replace", index=False)

    metadata = pd.DataFrame([
        {"key": "api_base_url", "value": API_BASE_URL},
        {"key": "retrieved_at_utc", "value": snapshot["retrieved_at_utc"]},
        {"key": "raw_record_count", "value": str(len(df))},
        {"key": "complete_record_count", "value": str(len(df_complete))},
        {"key": "required_complete_fields", "value": json.dumps(REQUIRED_COMPLETE_FIELDS)},
    ])
    metadata.to_sql("metadata", conn, if_exists="replace", index=False)

print(f"Saved CSV to {CSV_PATH}")
print(f"Saved SQLite database to {SQLITE_PATH}")


Saved CSV to data/ufc_fighters_complete_stats.csv
Saved SQLite database to data/ufc_fighters.sqlite


## Sanity-check the database


In [16]:
with sqlite3.connect(SQLITE_PATH) as conn:
    display(pd.read_sql_query("SELECT * FROM metadata", conn))
    display(pd.read_sql_query("SELECT COUNT(*) AS n_complete_fighters FROM fighters_complete", conn))


,key,value
0,api_base_url,https://ufc-api-theta.vercel.app/mma-api/fighters
1,retrieved_at_utc,2026-05-07T16:27:44.336733+00:00
2,raw_record_count,4262
3,complete_record_count,3955
4,required_complete_fields,"[""name"", ""height"", ""weight"", ""n_win"", ""n_loss""..."


,n_complete_fighters
0,3955


In [17]:
# Example: top fighters by recorded wins among complete-stat records.
with sqlite3.connect(SQLITE_PATH) as conn:
    top_wins = pd.read_sql_query(
        """
        SELECT name, n_win, n_loss, n_draw, sig_str_land_pM, td_avg, sub_avg
        FROM fighters_complete
        ORDER BY n_win DESC, n_loss ASC
        LIMIT 20
        """,
        conn,
    )

top_wins


,name,n_win,n_loss,n_draw,sig_str_land_pM,td_avg,sub_avg
0,Travis Fulton,253,53,10,0.00,0.00,0.0
1,Dan Severn,101,19,1,0.00,0.00,0.0
2,Jeremy Horn,91,22,5,1.19,1.83,1.1
3,Travis Wiuff,75,21,0,0.48,4.84,1.2
4,Luis Santos,63,10,1,4.40,0.00,0.0
5,Aleksei Oleinik,60,17,1,3.29,1.82,1.9
6,Jeff Monson,60,26,1,0.97,1.06,0.7
7,Yuki Kondo,60,33,9,1.14,0.76,0.5
8,Ikuhisa Minowa,60,42,8,0.88,1.82,1.9
9,Alexander Shlemenko,56,9,0,4.36,0.00,0.0


In [18]:
# Example: basic summary statistics for numeric fighter stats.
df_complete[NUMERIC_FIELDS].describe().T.round(3)


,count,mean,std,min,25%,50%,75%,max
n_win,3955.0,13.008,9.295,0.0,7.000,11.00,17.00,253.00
n_loss,3955.0,5.891,5.025,0.0,3.000,5.00,8.00,83.00
n_draw,3955.0,0.271,0.835,0.0,0.000,0.00,0.00,11.00
sig_str_land_pM,3955.0,2.621,1.927,0.0,1.200,2.54,3.74,20.55
sig_str_land_pct,3955.0,0.380,0.188,0.0,0.320,0.42,0.50,1.00
sig_str_abs_pM,3955.0,3.355,2.784,0.0,1.895,3.07,4.38,52.50
sig_str_def_pct,3955.0,0.452,0.201,0.0,0.400,0.51,0.58,1.00
td_avg,3955.0,1.330,1.967,0.0,0.000,0.73,2.00,32.14
td_land_pct,3955.0,0.280,0.284,0.0,0.000,0.26,0.46,1.00
td_def_pct,3955.0,0.423,0.337,0.0,0.000,0.50,0.68,1.00


## Query template

Use this cell as a template for future analysis.


In [20]:
query = """
SELECT
    name,
    height,
    weight,
    n_win,
    n_loss,
    sig_str_land_pM, 
    sig_str_land_pct, 
    sig_str_abs_pM,
    sig_str_def_pct,
    td_avg, 
    td_land_pct, 
    td_def_pct, 
    sub_avg
FROM fighters_complete
ORDER BY sig_str_land_pM DESC
LIMIT 10;
"""

with sqlite3.connect(SQLITE_PATH) as conn:
    result = pd.read_sql_query(query, conn)

result


,name,height,weight,n_win,n_loss,sig_str_land_pM,sig_str_land_pct,sig_str_abs_pM,sig_str_def_pct,td_avg,td_land_pct,td_def_pct,sub_avg
0,Yuneisy Duben,"5' 4""",125 lbs.,6,0,20.55,0.64,11.51,0.41,0.00,0.00,0.00,0.0
1,Rex Richards,"6' 5""",265 lbs.,7,2,17.65,0.58,5.29,0.72,0.00,0.00,0.00,0.0
2,Tallison Teixeira,"6' 7""",258 lbs.,7,0,14.87,0.54,9.74,0.58,0.00,0.00,0.00,0.0
3,Islam Dulatov,"6' 3""",170 lbs.,11,1,14.27,0.56,5.49,0.51,5.49,1.00,0.00,0.0
4,JR Coughran,"5' 6""",145 lbs.,4,1,12.29,0.57,6.80,0.45,0.00,0.00,0.75,0.0
5,Kim Couture,"5' 8""",135 lbs.,3,8,12.12,0.80,2.31,0.69,0.00,0.00,0.00,0.0
6,Ramazan Kuramagomedov,"6' 1""",170 lbs.,8,0,11.67,0.67,9.13,0.54,1.00,0.14,0.00,0.0
7,Shannon Clark,"5' 5""",125 lbs.,5,1,11.51,0.58,20.55,0.35,0.00,0.00,0.00,0.0
8,Carli Judice,"5' 7""",125 lbs.,3,2,11.23,0.54,10.10,0.52,1.00,0.25,0.70,0.0
9,Taiyilake Nueraji,"6' 2""",170 lbs.,9,1,11.01,0.48,7.39,0.58,0.00,0.00,0.00,0.0


## Notes for adapting the completeness rule

If you decide that date of birth or stance should not be required, edit this line above:

```python
REQUIRED_COMPLETE_FIELDS = PROFILE_FIELDS + NUMERIC_FIELDS
```

For example, to require only the numeric statistics and the fighter name:

```python
REQUIRED_COMPLETE_FIELDS = ["name"] + NUMERIC_FIELDS
```
